In [5]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os

dynamic_lc = xr.open_dataset("/projects/prjs2023/chapter3/lpjml_output/2026_01_29_baseline_run/monthlybackground/output/fpc.nc").FPC.sel(time = "1960-01")[0,1:]
dynamic_lc

aridity = xr.open_dataset(
    "/projects/prjs2023/chapter3/aridity/aridity_mean_0_1_degree.nc"
).__xarray_dataarray_variable__.sel(lat = slice(-55.9,83.75))

aridity['lon'] = dynamic_lc['lon']
aridity['lat'] = dynamic_lc['lat']
aridity

<xarray.DataArray '__xarray_dataarray_variable__' (lat: 1397, lon: 3600)> Size: 20MB
[5029200 values with dtype=float32]
Coordinates:
  * lat      (lat) float64 11kB -55.95 -55.85 -55.75 ... 83.45 83.55 83.65
  * lon      (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.9 180.0
    expver   int32 4B ...
    time     object 8B 1960-01-16 00:00:00

In [6]:
pft_names_main = [
    "tropical broadleaved evergreen tree",
    "tropical broadleaved raingreen tree",
    "temperate needleleaved evergreen tree",
    "temperate broadleaved evergreen tree",
    "temperate broadleaved summergreen tree",
    "boreal needleleaved evergreen tree",
    "boreal broadleaved summergreen tree",
    "boreal needleleaved summergreen tree"
]

In [ ]:
import numpy as np
import xarray as xr
import os

# ----------------------------
# PARAMETERS
# ----------------------------
patch_height = 8
patch_width  = 8
buffer_deg   = 1.0
aridity_thr  = 1.0
top_n        = 10
min_dist_cells = 20

# ----------------------------
# FUNCTION: find distinct hotspots
# ----------------------------
def find_top_n_hotspots_dist(patch_sum, n=10, min_dist_cells=20):
    data = patch_sum.values
    flat = data.flatten()

    valid_idx = np.where(~np.isnan(flat))[0]
    sorted_idx = valid_idx[np.argsort(flat[valid_idx])[::-1]]

    hotspots = []

    for idx in sorted_idx:
        i, j = np.unravel_index(idx, data.shape)

        too_close = False
        for (ii, jj, _) in hotspots:
            if abs(i - ii) < min_dist_cells and abs(j - jj) < min_dist_cells:
                too_close = True
                break

        if not too_close:
            hotspots.append((i, j, data[i, j]))

        if len(hotspots) == n:
            break

    return hotspots


# ----------------------------
# CLIMATE FILES
# ----------------------------
climate_data = {
    "wind":  { "var": "sfcwind", "file": "/projects/prjs2023/chapter3/lpjml_input/gswp3-w5e5_obsclim_sfcwind_global_daily_1901_2019_version_2021-09-XX.nc" },
    "temp":  { "var": "tas",     "file": "/projects/prjs2023/chapter3/lpjml_input/gswp3-w5e5_obsclim_tas_global_daily_1901_2019_version_2021-09-XX_degreesC.nc" },
    "prec":  { "var": "pr",      "file": "/projects/prjs2023/chapter3/lpjml_input/gswp3-w5e5_obsclim_pr_global_daily_1901_2019_version_2021-09-XX_mm_day.nc" },
    "lwdown":{ "var": "rlds",    "file": "/projects/prjs2023/chapter3/lpjml_input/gswp3-w5e5_obsclim_rlds_global_daily_1901_2019_version_2021-09-XX.nc" },
    "swdown":{ "var": "rsds",    "file": "/projects/prjs2023/chapter3/lpjml_input/gswp3-w5e5_obsclim_rsds_global_daily_1901_2019_version_2021-09-XX.nc" },
    "vpd":   { "var": "VPD",     "file": "/projects/prjs2023/chapter3/lpjml_input/VPD_W5E5_daily_05deg_1901_2019.nc" },
    "soil":  { "var": "soilcode","file": "/projects/prjs2023/chapter3/lpjml_input/regridded_global_usda_soilmap.nc" }
}

base_out = "/projects/prjs2023/chapter3/lpjml_input/run_for_10_locations"


# ----------------------------
# MAIN LOOP OVER PFTs
# ----------------------------
for i, pft_name in enumerate(pft_names_main):

    print(f"\n🌱 Processing: {pft_name}")

    fpc_da = dynamic_lc.isel(pft=i)
    fpc_wl = fpc_da.where(aridity < aridity_thr)

    # Moving-window sum
    patch_sum = fpc_wl.rolling(
        lat=patch_height,
        lon=patch_width,
        center=True
    ).construct({"lat": "lat_patch", "lon": "lon_patch"}).sum(dim=("lat_patch", "lon_patch"))

    if np.all(np.isnan(patch_sum)):
        print("⚠️ No valid data")
        continue

    # ----------------------------
    # 🔥 GET TOP 10 HOTSPOTS
    # ----------------------------
    hotspots = find_top_n_hotspots_dist(
        patch_sum,
        n=top_n,
        min_dist_cells=min_dist_cells
    )

    print(f"Top {len(hotspots)} hotspots:")

    for k, (ii, jj, val) in enumerate(hotspots, start=1):

        lat_center = patch_sum.lat[ii].item()
        lon_center = patch_sum.lon[jj].item()

        print(f"{k}: value={val:.3f}, lat={lat_center:.2f}, lon={lon_center:.2f}")

        # ----------------------------
        # DEFINE PATCH BOUNDS
        # ----------------------------
        lat_res = fpc_da.lat[1] - fpc_da.lat[0]
        lon_res = fpc_da.lon[1] - fpc_da.lon[0]

        lat0 = lat_center - (patch_height / 2) * lat_res
        lat1 = lat_center + (patch_height / 2) * lat_res
        lon0 = lon_center - (patch_width / 2) * lon_res
        lon1 = lon_center + (patch_width / 2) * lon_res

        # buffer
        latb0 = max(lat0 - buffer_deg, fpc_da.lat.min().item())
        latb1 = min(lat1 + buffer_deg, fpc_da.lat.max().item())
        lonb0 = max(lon0 - buffer_deg, fpc_da.lon.min().item())
        lonb1 = min(lon1 + buffer_deg, fpc_da.lon.max().item())

        # ----------------------------
        # CREATE OUTPUT FOLDER
        # ----------------------------
        patch_folder = os.path.join(
            base_out,
            pft_name.replace(" ", "_"),
            f"patch_{k}"
        )
        os.makedirs(patch_folder, exist_ok=True)

        # ----------------------------
        # SAVE CLIMATE DATA
        # ----------------------------
        for var_name, details in climate_data.items():

            ds = xr.open_dataset(details["file"])
            ds_var = ds[details["var"]]

            # soil = core, others = buffered
            if var_name == "soil":
                lat_min, lat_max = lat0, lat1
                lon_min, lon_max = lon0, lon1
            else:
                lat_min, lat_max = latb0, latb1
                lon_min, lon_max = lonb0, lonb1

            # Handle lat direction
            if ds_var.lat.values[0] < ds_var.lat.values[-1]:
                ds_sub = ds_var.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
            else:
                ds_sub = ds_var.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))

            if ds_sub.lat.size == 0 or ds_sub.lon.size == 0:
                print(f"❌ Empty slice for {var_name} (patch {k})")
                ds.close()
                continue

            out_file = os.path.join(patch_folder, f"{var_name}.nc")
            ds_sub.to_netcdf(out_file)
            ds.close()

        print(f"✅ Saved patch {k}")


🌱 Processing: tropical broadleaved evergreen tree
Top 10 hotspots:
1: value=59.003, lat=-14.85, lon=-39.95
✅ Saved patch 1
2: value=58.681, lat=-19.85, lon=-58.15
✅ Saved patch 2
3: value=57.705, lat=-16.85, lon=-39.85
✅ Saved patch 3
4: value=57.639, lat=20.65, lon=-88.55
✅ Saved patch 4
5: value=57.349, lat=-18.55, lon=-62.25


In [ ]:
pft_names_main = [
    "tropical broadleaved evergreen tree",
    "tropical broadleaved raingreen tree",
    "temperate needleleaved evergreen tree",
    "temperate broadleaved evergreen tree",
    "temperate broadleaved summergreen tree",
    "boreal needleleaved evergreen tree",
    "boreal broadleaved summergreen tree",
    "boreal needleleaved summergreen tree"
]

patch_height = 8
patch_width  = 8          # 8 × 8 = 64 cells
buffer       = 1.0        # degrees
aridity_thr  = 1.0

import matplotlib.pyplot as plt
import numpy as np

# Function to compute cell edges for pcolormesh
def compute_edges(vals):
    edges = np.zeros(len(vals) + 1)
    edges[1:-1] = (vals[:-1] + vals[1:]) / 2
    edges[0] = vals[0] - (vals[1] - vals[0]) / 2
    edges[-1] = vals[-1] + (vals[-1] - vals[-2]) / 2
    return edges

# Multi-panel setup
ncols = 3  # moving sum, FPC, aridity
nrows = len(pft_names_main)
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3*nrows), constrained_layout=True)

for i, pft_name in enumerate(pft_names_main):
    # Select FPC for this PFT and mask energy-limited regions
    fpc_da = dynamic_lc.isel(pft=i)
    fpc_wl = fpc_da.where(aridity < aridity_thr)

    # Moving-window sum
    patch_sum = fpc_wl.rolling(
        lat=patch_height,
        lon=patch_width,
        center=True
    ).construct({"lat": "lat_patch", "lon": "lon_patch"}).sum(dim=("lat_patch", "lon_patch"))

    if np.all(np.isnan(patch_sum)):
        for j in range(ncols):
            axes[i,j].axis("off")
        continue

    # -------------------------------
    # 🔥 HOTSPOT SELECTION (UPDATED)
    # -------------------------------
    vals = patch_sum.values.flatten()
    valid_idx = np.where(~np.isnan(vals))[0]

    if len(valid_idx) == 0:
        for j in range(ncols):
            axes[i,j].axis("off")
        continue

    if pft_name == "boreal needleleaved summergreen tree":
        # Sort descending
        sorted_idx = valid_idx[np.argsort(vals[valid_idx])[::-1]]

        if len(sorted_idx) > 1:
            chosen_idx = sorted_idx[1]  # second best
        else:
            chosen_idx = sorted_idx[0]  # fallback
    else:
        # Best location
        chosen_idx = valid_idx[np.argmax(vals[valid_idx])]

    i_center, j_center = np.unravel_index(chosen_idx, patch_sum.shape)

    # -------------------------------

    lat_center = patch_sum.lat[i_center].item()
    lon_center = patch_sum.lon[j_center].item()

    # Core patch bounds
    lat_res = fpc_da.lat[1] - fpc_da.lat[0]
    lon_res = fpc_da.lon[1] - fpc_da.lon[0]
    lat0 = lat_center - (patch_height / 2) * lat_res
    lat1 = lat_center + (patch_height / 2) * lat_res
    lon0 = lon_center - (patch_width / 2) * lon_res
    lon1 = lon_center + (patch_width / 2) * lon_res

    # Zoom bounds
    latb0 = max(lat0 - buffer, fpc_da.lat.min().item())
    latb1 = min(lat1 + buffer, fpc_da.lat.max().item())
    lonb0 = max(lon0 - buffer, fpc_da.lon.min().item())
    lonb1 = min(lon1 + buffer, fpc_da.lon.max().item())

    # Slice data
    patch_sum_zoom = patch_sum.sel(lat=slice(latb0, latb1), lon=slice(lonb0, lonb1))
    fpc_zoom       = fpc_da.sel(lat=slice(latb0, latb1), lon=slice(lonb0, lonb1))
    arid_zoom      = aridity.sel(lat=slice(latb0, latb1), lon=slice(lonb0, lonb1))

    # Edges
    lat_edges = compute_edges(patch_sum_zoom.lat.values)
    lon_edges = compute_edges(patch_sum_zoom.lon.values)

    # --- Plotting ---
    # Moving-window sum
    ax = axes[i,0]
    im0 = ax.pcolormesh(lon_edges, lat_edges, patch_sum_zoom.values, cmap="YlGn")
    ax.plot([lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color="red", lw=2)
    ax.set_title(f"{pft_name}\nMoving-window sum")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    fig.colorbar(im0, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="Total FPC")

    # Original FPC
    ax = axes[i,1]
    im1 = ax.pcolormesh(lon_edges, lat_edges, fpc_zoom.values, cmap="YlGn", vmin=0, vmax=1)
    ax.plot([lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color="red", lw=2)
    ax.set_title("Original FPC")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    fig.colorbar(im1, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="FPC")

    # Aridity
    ax = axes[i,2]
    im2 = ax.pcolormesh(lon_edges, lat_edges, arid_zoom.values, cmap="BrBG", vmin=0, vmax=2)
    ax.plot([lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color="red", lw=2)
    ax.set_title("Aridity")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    fig.colorbar(im2, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="Aridity index")

# Turn off unused axes
for ax_row in axes.flatten()[len(pft_names_main)*ncols:]:
    ax_row.axis("off")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Function to compute cell edges for pcolormesh
def compute_edges(vals):
    edges = np.zeros(len(vals) + 1)
    edges[1:-1] = (vals[:-1] + vals[1:]) / 2
    edges[0] = vals[0] - (vals[1] - vals[0]) / 2
    edges[-1] = vals[-1] + (vals[-1] - vals[-2]) / 2
    return edges

# Multi-panel setup
ncols = 3  # global map, zoomed FPC, zoomed aridity
nrows = len(pft_names_main)
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3*nrows), constrained_layout=True)

for i, pft_name in enumerate(pft_names_main):
    fpc_da = dynamic_lc.isel(pft=i)
    fpc_wl = fpc_da.where(aridity < aridity_thr)

    # Moving-window sum
    patch_sum = fpc_wl.rolling(
        lat=patch_height,
        lon=patch_width,
        center=True
    ).construct({"lat": "lat_patch", "lon": "lon_patch"}).sum(dim=("lat_patch", "lon_patch"))

    if np.all(np.isnan(patch_sum)):
        for j in range(ncols):
            axes[i,j].axis("off")
        continue

    # -----------------------------------
    # 🔥 UPDATED HOTSPOT SELECTION
    # -----------------------------------
    vals = patch_sum.values.flatten()
    valid_idx = np.where(~np.isnan(vals))[0]

    if len(valid_idx) == 0:
        for j in range(ncols):
            axes[i,j].axis("off")
        continue

    if pft_name == "boreal needleleaved summergreen tree":
        # Sort descending → take second best
        sorted_idx = valid_idx[np.argsort(vals[valid_idx])[::-1]]

        if len(sorted_idx) > 1:
            chosen_idx = sorted_idx[1]
        else:
            chosen_idx = sorted_idx[0]
    else:
        # Default: best
        chosen_idx = valid_idx[np.argmax(vals[valid_idx])]

    i_center, j_center = np.unravel_index(chosen_idx, patch_sum.shape)
    # -----------------------------------

    lat_center = patch_sum.lat[i_center].item()
    lon_center = patch_sum.lon[j_center].item()

    # Core patch bounds
    lat_res = fpc_da.lat[1] - fpc_da.lat[0]
    lon_res = fpc_da.lon[1] - fpc_da.lon[0]
    lat0 = lat_center - (patch_height / 2) * lat_res
    lat1 = lat_center + (patch_height / 2) * lat_res
    lon0 = lon_center - (patch_width / 2) * lon_res
    lon1 = lon_center + (patch_width / 2) * lon_res

    # Zoom bounds
    buffer_deg = 1.0
    latb0 = max(lat0 - buffer_deg, fpc_da.lat.min().item())
    latb1 = min(lat1 + buffer_deg, fpc_da.lat.max().item())
    lonb0 = max(lon0 - buffer_deg, fpc_da.lon.min().item())
    lonb1 = min(lon1 + buffer_deg, fpc_da.lon.max().item())

    # Slice data
    fpc_zoom  = fpc_da.sel(lat=slice(latb0, latb1), lon=slice(lonb0, lonb1))
    arid_zoom = aridity.sel(lat=slice(latb0, latb1), lon=slice(lonb0, lonb1))

    # Edges
    lat_edges_zoom = compute_edges(fpc_zoom.lat.values)
    lon_edges_zoom = compute_edges(fpc_zoom.lon.values)
    lat_edges_global = compute_edges(fpc_da.lat.values)
    lon_edges_global = compute_edges(fpc_da.lon.values)

    # --- Panel 1: Global FPC ---
    ax = axes[i,0]
    im0 = ax.pcolormesh(lon_edges_global, lat_edges_global, fpc_da.values, cmap="YlGn", vmin=0, vmax=1)
    ax.scatter(lon_center, lat_center, color="red", s=80, marker="x", label="Hotspot")
    ax.set_title(f"{pft_name}\nGlobal FPC")
    ax.set_xlabel("Lon"); ax.set_ylabel("Lat")
    ax.legend()
    fig.colorbar(im0, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="FPC")

    # --- Panel 2: Zoomed FPC ---
    ax = axes[i,1]
    im1 = ax.pcolormesh(lon_edges_zoom, lat_edges_zoom, fpc_zoom.values, cmap="YlGn", vmin=0, vmax=1)
    ax.plot([lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color="red", lw=2)
    ax.set_title("Zoomed FPC")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    fig.colorbar(im1, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="FPC")

    # --- Panel 3: Zoomed Aridity ---
    ax = axes[i,2]
    im2 = ax.pcolormesh(lon_edges_zoom, lat_edges_zoom, arid_zoom.values, cmap="BrBG", vmin=0, vmax=2)
    ax.plot([lon0, lon1, lon1, lon0, lon0],
            [lat0, lat0, lat1, lat1, lat0],
            color="red", lw=2)
    ax.set_title("Zoomed Aridity")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    fig.colorbar(im2, ax=ax, orientation="vertical", fraction=0.05, pad=0.02, label="Aridity index")

# Turn off unused axes
for ax_row in axes.flatten()[len(pft_names_main)*ncols:]:
    ax_row.axis("off")

plt.show()

In [ ]:
def find_top_n_hotspots_dist(patch_sum, n=10, min_dist_cells=20):
    data = patch_sum.values
    flat = data.flatten()

    valid_idx = np.where(~np.isnan(flat))[0]
    sorted_idx = valid_idx[np.argsort(flat[valid_idx])[::-1]]

    hotspots = []

    for idx in sorted_idx:
        i, j = np.unravel_index(idx, data.shape)

        too_close = False
        for (ii, jj, _) in hotspots:
            if abs(i - ii) < min_dist_cells and abs(j - jj) < min_dist_cells:
                too_close = True
                break

        if not too_close:
            hotspots.append((i, j, data[i, j]))

        if len(hotspots) == n:
            break

    return hotspots


if pft_name == "boreal needleleaved summergreen tree":
    hotspots = find_top_n_hotspots_dist(
        patch_sum,
        n=10,
        min_dist_cells=20   
    )

    print(f"\nTop 10 DISTINCT hotspots for {pft_name}:")
    for k, (ii, jj, val) in enumerate(hotspots):
        lat = patch_sum.lat[ii].item()
        lon = patch_sum.lon[jj].item()
        print(f"{k+1}: value={val:.3f}, lat={lat:.2f}, lon={lon:.2f}")

    i_center, j_center, _ = hotspots[0]
else:
    i_center, j_center = np.unravel_index(np.nanargmax(patch_sum.values), patch_sum.shape)

In [ ]:

climate_data = {
    "wind":  { "var": "sfcwind", "file": "/archive/depfg/ruiij001/Chapter_2/W5E5/gswp3-w5e5_obsclim_sfcwind_global_daily_1901_2019_version_2021-09-XX.nc" },
    "temp":  { "var": "tas",     "file": "/archive/depfg/ruiij001/Chapter_2/LPJmL_input_final/gswp3-w5e5_obsclim_tas_global_daily_1901_2019_version_2021-09-XX_degreesC.nc" },
    "prec":  { "var": "pr",      "file": "/archive/depfg/ruiij001/Chapter_2/LPJmL_input_final/gswp3-w5e5_obsclim_pr_global_daily_1901_2019_version_2021-09-XX_mm_day.nc" },
    "lwdown":{ "var": "rlds",    "file": "/archive/depfg/ruiij001/Chapter_2/LPJmL_input_final/gswp3-w5e5_obsclim_rlds_global_daily_1901_2019_version_2021-09-XX.nc" },
    "swdown":{ "var": "rsds",    "file": "/archive/depfg/ruiij001/Chapter_2/LPJmL_input_final/gswp3-w5e5_obsclim_rsds_global_daily_1901_2019_version_2021-09-XX.nc" },
    "vpd":   { "var": "VPD",     "file": "/archive/depfg/ruiij001/VPD_W5E5_daily/VPD_W5E5_daily_05deg_1901_2019.nc" },
    "soil":  { "var": "soilcode","file": "/archive/depfg/ruiij001/Chapter_2/LPJmL_input_regrid_01_degrees/new_2025_01_14/regridded_global_usda_soilmap.nc" }
}

for k, (ii, jj, val) in enumerate(hotspots, start=1):
    lat_center = patch_sum.lat[ii].item()
    lon_center = patch_sum.lon[jj].item()

    lat_res = fpc_da.lat[1] - fpc_da.lat[0]
    lon_res = fpc_da.lon[1] - fpc_da.lon[0]

    # Core patch bounds
    lat0 = lat_center - (patch_height / 2) * lat_res
    lat1 = lat_center + (patch_height / 2) * lat_res
    lon0 = lon_center - (patch_width / 2) * lon_res
    lon1 = lon_center + (patch_width / 2) * lon_res

    # Buffer bounds
    latb0 = max(lat0 - buffer_deg, fpc_da.lat.min().item())
    latb1 = min(lat1 + buffer_deg, fpc_da.lat.max().item())
    lonb0 = max(lon0 - buffer_deg, fpc_da.lon.min().item())
    lonb1 = min(lon1 + buffer_deg, fpc_da.lon.max().item())

    base_out = "/eejit/home/ruiij001/scripts/Chapter_3/2026_03_24_find_best_locations"
    # Create folder for this patch
    patch_folder = os.path.join(base_out,
                                pft_name.replace(" ", "_"),
                                f"patch_{k}")
    os.makedirs(patch_folder, exist_ok=True)

    # ----------------------------
    # Save climate variables
    # ----------------------------
    for var_name, details in climate_data.items():
        ds = xr.open_dataset(details["file"])
        ds_var = ds[details["var"]]

        # Use core bounds for soil, buffer bounds for others
        if var_name == "soil":
            lat_min, lat_max = lat0, lat1
            lon_min, lon_max = lon0, lon1
        else:
            lat_min, lat_max = latb0, latb1
            lon_min, lon_max = lonb0, lonb1

        # Handle ascending/descending latitude
        if ds_var.lat.values[0] < ds_var.lat.values[-1]:
            ds_sub = ds_var.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
        else:
            ds_sub = ds_var.sel(lat=slice(lat_max, lat_min), lon=slice(lon_min, lon_max))

        if ds_sub.lat.size == 0 or ds_sub.lon.size == 0:
            print(f"❌ Empty slice for {var_name} in patch {k}")
            ds.close()
            continue

        out_file = os.path.join(patch_folder, f"{var_name}.nc")
        ds_sub.to_netcdf(out_file)
        print(f"✅ Saved {out_file}")
        ds.close()